# **Training GLiNER model with MD-related simulation's description** ⚙️

[GLiNER2](https://github.com/fastino-ai/GLiNER2) unifies Named Entity Recognition, Text Classification, Structured Data Extraction, and Relation Extraction into a single 205M parameter model. It provides efficient CPU-based inference without requiring complex pipelines or external API dependencies.

- See Documentation: https://urchade.github.io/GLiNER/training.html
- See Preprint: https://arxiv.org/abs/2311.08526


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from gliner2 import GLiNER2
from gliner2.training.data import TrainingDataset
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

from mdner_llm.core.logger import create_logger
from mdner_llm.utils.evaluate_gliner2 import build_train_examples
from mdner_llm.utils.select_annotation_files import select_annotation_files
from mdner_llm.utils.visualize_annotations import (
    convert_ner_response_to_entities,
    visualize_llm_annotation,
)

/data/touami/mdner_llm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load models


In [2]:
# Load model with 205M parameters (base version)
base_extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
print(base_extractor)

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
GLiNER2(
  (encoder): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128011, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
    

In [3]:
# Load model with 340M parameters (large version)
# large_extractor = GLiNER2.from_pretrained("fastino/gliner2-large-v1")
# print(large_extractor)

## Define entities classes


In [4]:
# Class of entities
entities_class_with_description = {
    "MOL": "Molecule or chemical compound involved in the simulation",
    "SOFTNAME": "Molecular dynamics software used for the simulation",
    "SOFTVERS": "Version of the molecular dynamics software",
    "TEMP": "Simulation temperature, typically expressed in Kelvin or Celcius",
    "FFM": "Force field model used to describe interatomic interactions",
    "STIME": "Total simulation time or duration",
}

## Prepare training data

In [5]:
# Load annotations
selected_annotation_paths = select_annotation_files(
    annotations_dir=Path("../../annotations/v3"),
    nb_files=40,
    logger=create_logger()
)

2026-03-04 14:33:58 | INFO     | Selecting text to annotate from ../../annotations/v3.
2026-03-04 14:33:58 | SUCCESS  | Found 360 JSON annotation files.
2026-03-04 14:33:58 | INFO     | First annotation file path: ../../annotations/v3/zenodo_34415.json
2026-03-04 14:33:58 | SUCCESS  | Selected 40 interesting annotations successfully!


In [6]:
# Formate annotations for training
train_examples = build_train_examples(
    selected_annotation_paths,
    entity_descriptions=entities_class_with_description
)
print(f"Number of training examples: {len(train_examples)}")
print("First training example:")
print(f"Text: {train_examples[0].text}")
print(f"Entities: {train_examples[0].entities}")

Number of training examples: 40
First training example:
Text: POPC_AMBER_LIPID14_CaCl2_035Mol
MD simulation trajectory and related files for fully hydrated POPC bilayer with 0.35M CaCl2. The LIPID14 force field was used with Gromacs 5.0.3. Ions were described by AMBER99SB-ILDN force field. Conditions: T=298.15, 128 POPC molecules, 6400 tip3p waters (lipid/water 1:50), 35 Ca, 70 Cl. 200ns trajectory  (preceded by 5ns NPT equillibration) (2 files of 100ns).
THE TRAJECTORY "035M_CaCl2_POPC_AMB_100_200ns.xtc" IS CORRUPTED. FOR THE UNCORRUPTED FILE PLEASE FOLLOW THE LINK: https://zenodo.org/record/46234
This data is ran for the nmrlipids.blospot.fi project. More details from nmrlipids.blospot.fi and https://github.com/NMRLipids/nmrlipids.blogspot.fi
Entities: {'MOL': ['POPC', 'CaCl2', 'Ca', 'Cl'], 'FFM': ['LIPID14', 'AMBER99SB-ILDN', 'tip3p'], 'SOFTNAME': ['Gromacs'], 'SOFTVERS': ['5.0.3.'], 'TEMP': ['298.15'], 'STIME': ['200ns', '100ns']}


In [7]:
# Create dataset for training
train_dataset = TrainingDataset(train_examples)
train_dataset.print_stats()


GLiNER2 Training Dataset Statistics
Total examples: 40

Text lengths: min=607, max=7384, mean=1434.7

Task Distribution:
  entities_only: 40 (100.0%)

Entity Types (451 total mentions):
  MOL: 168
  FFM: 80
  SOFTNAME: 65
  TEMP: 55
  STIME: 50
  SOFTVERS: 33



In [8]:
# Validate the dataset
# This will raise an error if there are any issues with the dataset
train_dataset.validate(raise_on_error=True)
# We have 21 examples without a minimum of one instance of each entity type
# which is not ideal but we can still train the model with this "test" dataset.

ValidationError: Dataset validation failed: 11 invalid examples
  - Example 1: Entity description for 'SOFTVERS' but no entities of that type
  - Example 2: Entity description for 'SOFTVERS' but no entities of that type
  - Example 7: Entity description for 'SOFTVERS' but no entities of that type
  - Example 14: Entity description for 'SOFTVERS' but no entities of that type
  - Example 23: Entity description for 'SOFTVERS' but no entities of that type
  - Example 29: Entity description for 'SOFTVERS' but no entities of that type
  - Example 35: Entity description for 'SOFTVERS' but no entities of that type
  - Example 35: Entity description for 'TEMP' but no entities of that type
  - Example 35: Entity description for 'STIME' but no entities of that type
  - Example 36: Entity description for 'SOFTVERS' but no entities of that type
  ... and 15 more errors

In [9]:
# Split into train/validation
train_data, val_data, _ = train_dataset.split(
    train_ratio=0.6,
    val_ratio=0.4,
    test_ratio=0.0,
    shuffle=True,
    seed=42
)

In [10]:
# Save datasets
# train_data.save("../../data/ner_training/train.jsonl")
# val_data.save("../../data/ner_training/val.jsonl")

## Configure training

In [14]:
model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
config = TrainingConfig(
    output_dir="../../results/ner_model",
    experiment_name="gliner2_finetuned_50_descriptions",

    # Training steps
    num_epochs=10,
    max_steps=-1,

    # Batch size
    batch_size=8,
    eval_batch_size=16,
    gradient_accumulation_steps=1,

    # Learning rates
    encoder_lr=5e-6,  # Lower LR for fine-tuning
    task_lr=5e-4,

    # Early stopping
    early_stopping=False,
    early_stopping_patience=3,
    early_stopping_threshold=0.0,

    # Mixed precision
    fp16=True,

    # Checkpointing & Evaluation (saves when evaluating)
    eval_strategy="epoch",  # "epoch", "steps", or "no"
    metric_for_best="eval_loss",
    save_best=True,

    # Logging
    logging_steps=50,
)

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


## Train

In [15]:
trainer = GLiNER2Trainer(model, config)
results = trainer.train(
    train_data=train_data,
    eval_data=val_data
)

2026-03-04 14:35:01 - INFO - gliner2.training.trainer - Using device: cuda
2026-03-04 14:35:01 - INFO - gliner2.training.trainer - LoRA is disabled
Validating records: 100%|██████████| 24/24 [00:00<00:00, 13544.58record/s]
2026-03-04 14:35:01 - INFO - gliner2.training.trainer - ***** Running Training *****
2026-03-04 14:35:01 - INFO - gliner2.training.trainer -   Num examples = 24
2026-03-04 14:35:01 - INFO - gliner2.training.trainer -   Num epochs = 10
2026-03-04 14:35:01 - INFO - gliner2.training.trainer -   Batch size = 8
2026-03-04 14:35:01 - INFO - gliner2.training.trainer -   Gradient accumulation steps = 1
2026-03-04 14:35:01 - INFO - gliner2.training.trainer -   Effective batch size = 8
2026-03-04 14:35:01 - INFO - gliner2.training.trainer -   Total optimization steps = 30
2026-03-04 14:35:01 - INFO - gliner2.training.trainer -   Warmup steps = 3
2026-03-04 14:35:01 - INFO - gliner2.training.trainer -   Trainable parameters: 208,476,821 / 208,476,821 (100.00%)
Training:   7%|▋ 

## Load best model

In [16]:
model = GLiNER2.from_pretrained("../../results/ner_model/best")
model

The tokenizer you are loading from '../../results/ner_model/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


GLiNER2(
  (encoder): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128011, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-07, ele